# Hardware making with Components
## We will make Sender and receiver here

In [1]:
from sequence.kernel.timeline import Timeline
from sequence.topology.node import Node
from sequence.components.memory import Memory
from sequence.components.detector import Detector
from sequence.kernel.process import Process
from sequence.kernel.event import Event
import math

class Sender:
    def __init__(self, owner, memory_name):
        self.owner = owner
        self.memory = self.owner.components[memory_name]

    def start(self, period):
        # Debug: report call and relevant globals
        print(f"Sender.start called with period={period}, NUM_TRIALS={globals().get('NUM_TRIALS', 'undefined')}")
        process1 = Process(self.memory, "update_state", [[complex(math.sqrt(1/2)), complex(math.sqrt(1/2))]])
        process2 = Process(self.memory, "excite", ["node2"])
        for i in range(NUM_TRIALS):
            event1 = Event(i * period, process1)
            event2 = Event(i * period + (period / 2), process2)
            # Debug: print first few scheduled event times
            if i < 5:
                print(f'scheduling event1 at time {event1.time}, event2 at time {event2.time}')
            self.owner.timeline.schedule(event1)
            self.owner.timeline.schedule(event2)
        # Debug: show schedule counter after scheduling
        print(f'Scheduled events: {self.owner.timeline.schedule_counter}')

class Counter:
    def __init__(self):
        self.count = 0
        self.time = 0

    def trigger(self, detector, info):
        self.count += 1
        self.time = info['time']

class SenderNode(Node):
    def __init__(self, name, timeline):
        super().__init__(name, timeline)
        memory_name = name + ".memory"
        memory = Memory(memory_name, timeline, fidelity=1, frequency=0,
                        efficiency=1, coherence_time=1e-3, wavelength=500)
        self.add_component(memory)
        memory.add_receiver(self)

        self.sender = Sender(self, memory_name)

    def get(self, photon, **kwargs):
        self.send_qubit(kwargs['dst'], photon)

class ReceiverNode(Node):
    def __init__(self, name, timeline):
        super().__init__(name, timeline)

        detector_name = name + ".detector"
        detector = Detector(detector_name, timeline, efficiency=1)
        self.add_component(detector)
        self.set_first_component(detector_name)
        detector.owner = self

        self.counter = Counter()
        detector.attach(self.counter)

    def receive_qubit(self, src, qubit):
        self.components[self.first_component_name].get(qubit)
        self.counter.trigger(self.components[self.first_component_name], {'time': self.timeline.now()})

### Simple counter to count number of photons

### Start the timeline

In [2]:
from sequence.kernel.timeline import Timeline
tl = Timeline(10e12)

In [3]:
node1 = SenderNode("node1", tl)
node2 = ReceiverNode("node2", tl)
node1.set_seed(0)
node2.set_seed(1)

In [4]:
from sequence.components.optical_channel import QuantumChannel
qc = QuantumChannel("qc", tl, attenuation=0, distance=1e3)
qc.set_ends(node1, node2.name)

In [5]:
memory = node1.components['node1.memory']
memory.update_state([complex(0), complex(1)])
print("Memory state updated successfully")

Memory state updated successfully


In [6]:
print("node1 components:", node1.components)
print("Component types:", [type(c) for c in node1.components.values()])
memories = node1.get_components_by_type(Memory)
print("Memories in node1:", memories)


node1 components: {'node1.memory': <sequence.components.memory.Memory object at 0x0000026DF3CC4D70>}
Component types: [<class 'sequence.components.memory.Memory'>]
Memories in node1: []


### Sender Class for sending photons

In [7]:
FREQUENCY = 150 # Hz
NUM_TRIALS = 100
tl.init()
period = int(1e12 / FREQUENCY)
node1.sender.start(period)
tl.run()

Sender.start called with period=6666666666, NUM_TRIALS=100
scheduling event1 at time 0, event2 at time 3333333333.0
scheduling event1 at time 6666666666, event2 at time 9999999999.0
scheduling event1 at time 13333333332, event2 at time 16666666665.0
scheduling event1 at time 19999999998, event2 at time 23333333331.0
scheduling event1 at time 26666666664, event2 at time 29999999997.0
Scheduled events: 201


In [8]:
print("percent measured: {}%".format(100 * node2.counter.count / NUM_TRIALS))

percent measured: 100.0%


In [9]:
# Check if events were scheduled
print(f"Total events in timeline: {len(tl.events)}")
print(f"Detector count: {node2.counter.count}")
print(f"Detector trigger time: {node2.counter.time if node2.counter.count > 0 else 'Never triggered'}")

# Check if the detector was properly attached
detector = node2.components[node2.first_component_name]
print(f"Detector: {detector}")
print(f"Detector observers: {detector.observers if hasattr(detector, 'observers') else 'N/A'}")

Total events in timeline: 0
Detector count: 100
Detector trigger time: 663338333267.0
Detector: node2.detector
Detector observers: N/A
